# Strategy 4 — Composite Multi-Signal

Strategy 4 stops introducing new traversals and starts **fusing** the three you already have. Strategies 1 / 2 / 3 each produce a ranked list of recs with their own scoring scale (genre affinity counts in 5–60, director/actor counts in 2–40, collaborative combined-products in the hundreds). A naive weighted sum over raw scores would let Strategy 3 dominate regardless of weights — so this notebook discards magnitudes and keeps only **rank position**, then fuses via **Reciprocal Rank Fusion**:

> *score = sum over contributing strategies of `w_s / rank_s`*

A few notes on what this notebook is and isn't:

- **Rank-based, not score-based.** Each sub-strategy fetches its top `inner_k=15` recs. The composite remembers only that a rec was *#3 in genre* and *#7 in collaborative* — never the raw scores. RRF was chosen over min-max / score÷max because rank is scale-free and instantly comparable across strategies. Cell 12 shows what happens when you reweight; cell 13 shows what happens when you change `inner_k`.
- **`1/rank` formula, no smoothing constant.** Front-heavy: rec #1 contributes 1.0 to its strategy, #2 contributes 0.5, #15 contributes ~0.067. A rec that hits #1 in one strategy beats a rec that hits #5 in all three. Simpler than textbook RRF's `1/(60 + rank)` — clarity over robustness at notebook scale.
- **Strategy-bridged explainability.** Strategies 1 / 2 / 3 each carry one chain payload (genre-bridged, person-bridged, peer-bridged). Strategy 4 carries **`via_strategies`** (which sub-strategies surfaced this rec) **paired with `chains`** (one chain per contributing strategy, same order). A rec surfaced by both *genre* and *collaborative* exposes two chains — one explanation per signal that picked it.
- **Cold start out of scope.** If all three sub-strategies return `[]`, composite returns `[]`. The demographic-pivot fallback (the *"user with 1 rating still gets recs"* teaching moment) is filed as **T-22b**, not delivered here.
- **Anchor user: 196.** Same recurring anchor as notebooks 09 / 10 / 11 / 12 / 13.
- **Notebook structure: phase-by-phase as standalone Cypher first.** Cells 4–7 build each piece of the unified query in isolation; cells 8–10 reveal the assembled constant and show the wrapper returns the same rows.


In [1]:
# 1. Setup
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from graph import run_query, close_driver
from recommender import genre_affinity, director_actor_affinity, collaborative_filtering

ANCHOR_USER = 196
print(f"Anchor user: {ANCHOR_USER}")

Anchor user: 196


In [2]:
# 2. Three candidate pools — how much agreement do the signals show?
INNER_K = 15

genre_recs   = genre_affinity(ANCHOR_USER, k=INNER_K)
da_recs      = director_actor_affinity(ANCHOR_USER, k=INNER_K)
collab_recs  = collaborative_filtering(ANCHOR_USER, k=INNER_K)

genre_ids   = {r["movieId"] for r in genre_recs}
da_ids      = {r["movieId"] for r in da_recs}
collab_ids  = {r["movieId"] for r in collab_recs}

summary = pd.DataFrame([
    {"strategy": "genre",          "n_recs": len(genre_ids)},
    {"strategy": "director_actor", "n_recs": len(da_ids)},
    {"strategy": "collaborative",  "n_recs": len(collab_ids)},
])
print(f"Per-strategy pool sizes (k={INNER_K} each):")
print(summary.to_string(index=False))

overlap = pd.DataFrame([
    {"pair": "genre ∩ director_actor",          "n": len(genre_ids & da_ids)},
    {"pair": "genre ∩ collaborative",           "n": len(genre_ids & collab_ids)},
    {"pair": "director_actor ∩ collaborative",  "n": len(da_ids & collab_ids)},
    {"pair": "all three",                        "n": len(genre_ids & da_ids & collab_ids)},
    {"pair": "union (any signal)",               "n": len(genre_ids | da_ids | collab_ids)},
])
print("\nPairwise + three-way overlap and union:")
print(overlap.to_string(index=False))
print("\nReading: the three signals overlap less than you'd think. Most recs come from one strategy only;")
print("a rec surfaced by two or three is a much stronger endorsement — RRF will reward that automatically.")

Per-strategy pool sizes (k=15 each):
      strategy  n_recs
         genre      15
director_actor      15
 collaborative      15

Pairwise + three-way overlap and union:
                          pair  n
        genre ∩ director_actor  0
         genre ∩ collaborative  0
director_actor ∩ collaborative  1
                     all three  0
            union (any signal) 44

Reading: the three signals overlap less than you'd think. Most recs come from one strategy only;
a rec surfaced by two or three is a much stronger endorsement — RRF will reward that automatically.


## Let's build the query phase by phase

Before showing the unified `COMPOSITE_CYPHER` constant (cell 9) and calling the wrapper (cell 10), let's run each piece as its own Cypher block. Notebook 14's phases are different from 11 / 12 / 13's — instead of *"progressive steps of one query"*, each phase here is one of the three sub-strategies wrapped to attach **rank position**, plus a fourth merge phase:

- **Phase 1 — Strategy 1's contribution.** Run `GENRE_AFFINITY_CYPHER` with `$k=15`, then `collect` + `UNWIND range(...)` to attach 1-indexed rank. Output: `(movieId, title, year, strategy='genre', rank, chain)`.
- **Phase 2 — Strategy 2's contribution.** Same shape, using `DIRECTOR_ACTOR_CYPHER` and the `seed_chains` payload.
- **Phase 3 — Strategy 3's contribution.** Same shape, using `COLLABORATIVE_CYPHER` and the `peer_chains` payload.
- **Phase 4 — the merge.** Wrap phases 1 / 2 / 3 as three `CALL { ... }` subqueries in one Cypher block, union the three rank-lists, group by `movieId`, compute the weighted RRF score, build paired `via_strategies` + `chains`. This is the load-bearing cell: the reader sees the fusion happen as data, end-to-end.

Each phase cell ends with `RETURN` so it runs standalone. In cell 7's unified version, the `RETURN`s of phases 1 / 2 / 3 become the bodies of `CALL { }` subqueries, and Phase 4's `RETURN` becomes the final return of the whole composite query.

In [3]:
# 4. Phase 1 standalone — Strategy 1 (genre affinity) with rank attached
PHASE_1 = """
// Strategy 1 body — verbatim, $k bound to inner_k
MATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)
WHERE r.rating >= $threshold
WITH u, g, count(*) AS affinity
ORDER BY affinity DESC
LIMIT $top_n_genres
WITH u, collect({g: g, a: affinity}) AS scored_genres

UNWIND scored_genres AS sg
WITH u, scored_genres, sg.g AS top_g
MATCH (u)-[r2:RATED]->(seed:Movie)-[:IN_GENRE]->(top_g)
WHERE r2.rating >= $threshold
WITH u, scored_genres, collect(DISTINCT seed.title)[..5] AS seed_movies

UNWIND scored_genres AS sg
WITH u, seed_movies, sg.g AS top_g, sg.a AS top_a
MATCH (top_g)<-[:IN_GENRE]-(rec:Movie)
WHERE NOT EXISTS { (u)-[:RATED]->(rec) }
WITH rec, seed_movies,
     sum(top_a) AS score,
     collect(DISTINCT top_g.name) AS matched_genres
WITH rec.movieId AS movieId, rec.title AS title, rec.year AS year,
     score, matched_genres, seed_movies
ORDER BY score DESC, title ASC
LIMIT $inner_k

// Rank attachment — collect then enumerate via UNWIND range
WITH collect({
  movieId: movieId, title: title, year: year,
  chain: head(matched_genres) + ' → ' + head(seed_movies) + ' → ' + title
}) AS genre_list
UNWIND range(0, size(genre_list) - 1) AS i
RETURN genre_list[i].movieId AS movieId,
       genre_list[i].title   AS title,
       genre_list[i].year    AS year,
       'genre' AS strategy,
       i + 1   AS rank,
       genre_list[i].chain AS chain
"""

phase_1_rows = run_query(
    PHASE_1,
    {"userId": ANCHOR_USER, "threshold": 4.0, "top_n_genres": 3, "inner_k": INNER_K},
)
print(f"Phase 1 — {len(phase_1_rows)} ranked recs from Strategy 1 (genre):")
pd.DataFrame(phase_1_rows)[["rank", "title", "year", "chain"]]

Phase 1 — 15 ranked recs from Strategy 1 (genre):


,rank,title,year,chain
0,1,Brassed Off,1996.0,Comedy → Babe → Brassed Off
1,2,Cinema Paradiso,1988.0,Comedy → Babe → Cinema Paradiso
2,3,"Corrina, Corrina",1994.0,"Comedy → Babe → Corrina, Corrina"
3,4,Don Juan DeMarco,1995.0,Comedy → Babe → Don Juan DeMarco
4,5,I Like It Like That,1994.0,Comedy → Babe → I Like It Like That
5,6,Manhattan,1979.0,Comedy → Babe → Manhattan
6,7,Something to Talk About,1995.0,Comedy → Babe → Something to Talk About
7,8,Twelfth Night,1996.0,Comedy → Babe → Twelfth Night
8,9,What Happened Was...,1994.0,Comedy → Babe → What Happened Was...
9,10,Wings of Desire,1987.0,Comedy → Babe → Wings of Desire


In [4]:
# 5. Phase 2 standalone — Strategy 2 (director/actor) with rank attached
PHASE_2 = """
// Strategy 2 body — verbatim, $k bound to inner_k
MATCH (u:User {userId: $userId})-[r1:RATED]->(seed_d:Movie)-[:DIRECTED_BY]->(d:Director)
WHERE r1.rating >= $threshold
WITH u, d, count(seed_d) AS affinity, collect(DISTINCT seed_d.title) AS seed_titles
ORDER BY affinity DESC
LIMIT $top_n_directors
WITH u, collect({
  person_id: elementId(d), name: d.name, kind: 'director',
  affinity: affinity, seeds: seed_titles
}) AS top_dirs

MATCH (u)-[r2:RATED]->(seed_a:Movie)-[fa:FEATURES_ACTOR]->(a:Actor)
WHERE r2.rating >= $threshold AND fa.billing_order <= $max_billing
WITH u, top_dirs, a, count(seed_a) AS affinity, collect(DISTINCT seed_a.title) AS seed_titles
ORDER BY affinity DESC
LIMIT $top_n_actors
WITH u, top_dirs, collect({
  person_id: elementId(a), name: a.name, kind: 'actor',
  affinity: affinity, seeds: seed_titles
}) AS top_actors
WITH u, top_dirs + top_actors AS top_people

UNWIND top_people AS tp
MATCH (rec:Movie)
WHERE NOT EXISTS { (u)-[:RATED]->(rec) }
  AND (
    (tp.kind = 'director' AND EXISTS {
      MATCH (rec)-[:DIRECTED_BY]->(d2:Director)
      WHERE elementId(d2) = tp.person_id
    })
    OR
    (tp.kind = 'actor' AND EXISTS {
      MATCH (rec)-[fa2:FEATURES_ACTOR]->(a2:Actor)
      WHERE elementId(a2) = tp.person_id AND fa2.billing_order <= $max_billing
    })
  )
WITH rec, collect(tp) AS connecting_people
WITH rec,
     [p IN connecting_people | p.name + ' (' + p.kind + ')'] AS via_people,
     reduce(s = 0, p IN connecting_people | s + p.affinity) AS score,
     [p IN connecting_people | head(p.seeds) + ' → ' + p.name + ' (' + p.kind + ') → ' + rec.title][..3] AS seed_chains
WITH rec.movieId AS movieId, rec.title AS title, rec.year AS year,
     score, via_people, seed_chains
ORDER BY score DESC, title ASC
LIMIT $inner_k

// Rank attachment
WITH collect({
  movieId: movieId, title: title, year: year,
  chain: head(seed_chains)
}) AS da_list
UNWIND range(0, size(da_list) - 1) AS j
RETURN da_list[j].movieId AS movieId,
       da_list[j].title   AS title,
       da_list[j].year    AS year,
       'director_actor' AS strategy,
       j + 1            AS rank,
       da_list[j].chain AS chain
"""

phase_2_rows = run_query(
    PHASE_2,
    {"userId": ANCHOR_USER, "threshold": 4.0, "top_n_directors": 3, "top_n_actors": 5,
     "max_billing": 5, "inner_k": INNER_K},
)
print(f"Phase 2 — {len(phase_2_rows)} ranked recs from Strategy 2 (director/actor):")
pd.DataFrame(phase_2_rows)[["rank", "title", "year", "chain"]]

Phase 2 — 15 ranked recs from Strategy 2 (director/actor):


,rank,title,year,chain
0,1,Aladdin,1992.0,"Birdcage, The → Robin Williams (actor) → Aladdin"
1,2,Aladdin and the King of Thieves,1996.0,"Birdcage, The → Robin Williams (actor) → Aladd..."
2,3,Being Human,1993.0,"Birdcage, The → Robin Williams (actor) → Being..."
3,4,Dead Poets Society,1989.0,"Birdcage, The → Robin Williams (actor) → Dead ..."
4,5,Fathers' Day,1997.0,"Birdcage, The → Robin Williams (actor) → Fathe..."
5,6,Flubber,1997.0,"Birdcage, The → Robin Williams (actor) → Flubber"
6,7,Ghosts of Mississippi,1996.0,Stand by Me → Rob Reiner (director) → Ghosts o...
7,8,Good Will Hunting,1997.0,"Birdcage, The → Robin Williams (actor) → Good ..."
8,9,Jack,1996.0,"Birdcage, The → Robin Williams (actor) → Jack"
9,10,Jumanji,1995.0,"Birdcage, The → Robin Williams (actor) → Jumanji"


In [5]:
# 6. Phase 3 standalone — Strategy 3 (collaborative filtering) with rank attached
PHASE_3 = """
// Strategy 3 body — verbatim, $k bound to inner_k
MATCH (u:User {userId: $userId})-[r1:RATED]->(seed:Movie)<-[r2:RATED]-(peer:User)
WHERE r1.rating >= $threshold
  AND r2.rating >= $threshold
  AND peer <> u
WITH u, peer, count(DISTINCT seed) AS overlap, collect(DISTINCT seed.title) AS shared_titles
WHERE overlap >= $min_overlap
WITH u, peer, overlap, shared_titles
ORDER BY overlap DESC
LIMIT $top_n_peers
WITH u, collect({
  peer_id: peer.userId, overlap: overlap, shared: shared_titles
}) AS top_peers

UNWIND top_peers AS tp
MATCH (peer:User {userId: tp.peer_id})-[r3:RATED]->(rec:Movie)
WHERE r3.rating >= $threshold
  AND NOT EXISTS { (u)-[:RATED]->(rec) }
WITH rec, tp, r3.rating AS peer_rating

WITH rec, collect({
  peer_id: tp.peer_id, overlap: tp.overlap,
  peer_rating: peer_rating, shared: tp.shared
}) AS contributing
WITH rec,
     reduce(s = 0.0, c IN contributing | s + (c.overlap * c.peer_rating)) AS score,
     [c IN contributing | {peer_id: c.peer_id, overlap: c.overlap}] AS via_peers,
     [c IN contributing | head(c.shared) + ' → peer ' + toString(c.peer_id) + ' → ' + rec.title][..3] AS peer_chains
WITH rec.movieId AS movieId, rec.title AS title, rec.year AS year,
     score, via_peers, peer_chains
ORDER BY score DESC, title ASC
LIMIT $inner_k

// Rank attachment
WITH collect({
  movieId: movieId, title: title, year: year,
  chain: head(peer_chains)
}) AS cf_list
UNWIND range(0, size(cf_list) - 1) AS m
RETURN cf_list[m].movieId AS movieId,
       cf_list[m].title   AS title,
       cf_list[m].year    AS year,
       'collaborative' AS strategy,
       m + 1           AS rank,
       cf_list[m].chain AS chain
"""

phase_3_rows = run_query(
    PHASE_3,
    {"userId": ANCHOR_USER, "threshold": 4.0, "min_overlap": 3, "top_n_peers": 25,
     "inner_k": INNER_K},
)
print(f"Phase 3 — {len(phase_3_rows)} ranked recs from Strategy 3 (collaborative):")
pd.DataFrame(phase_3_rows)[["rank", "title", "year", "chain"]]

Phase 3 — 15 ranked recs from Strategy 3 (collaborative):


,rank,title,year,chain
0,1,Star Wars,1977.0,Babe → peer 303 → Star Wars
1,2,Back to the Future,1985.0,Babe → peer 303 → Back to the Future
2,3,Raiders of the Lost Ark,1981.0,Babe → peer 303 → Raiders of the Lost Ark
3,4,Amadeus,1984.0,Babe → peer 303 → Amadeus
4,5,Fargo,1996.0,Babe → peer 303 → Fargo
5,6,"Shawshank Redemption, The",1994.0,"Babe → peer 303 → Shawshank Redemption, The"
6,7,"Empire Strikes Back, The",1980.0,"Babe → peer 303 → Empire Strikes Back, The"
7,8,Return of the Jedi,1983.0,Babe → peer 303 → Return of the Jedi
8,9,Monty Python and the Holy Grail,1974.0,Babe → peer 303 → Monty Python and the Holy Grail
9,10,Pulp Fiction,1994.0,Babe → peer 303 → Pulp Fiction


In [6]:
# 7. Phase 4 standalone — merge: union the three rank-lists, group by movieId, weighted RRF
PHASE_4 = """
// Subquery 1 — Strategy 1 (genre affinity)
CALL {
  MATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)
  WHERE r.rating >= $threshold
  WITH u, g, count(*) AS affinity
  ORDER BY affinity DESC
  LIMIT $top_n_genres
  WITH u, collect({g: g, a: affinity}) AS scored_genres
  UNWIND scored_genres AS sg
  WITH u, scored_genres, sg.g AS top_g
  MATCH (u)-[r2:RATED]->(seed:Movie)-[:IN_GENRE]->(top_g)
  WHERE r2.rating >= $threshold
  WITH u, scored_genres, collect(DISTINCT seed.title)[..5] AS seed_movies
  UNWIND scored_genres AS sg
  WITH u, seed_movies, sg.g AS top_g, sg.a AS top_a
  MATCH (top_g)<-[:IN_GENRE]-(rec:Movie)
  WHERE NOT EXISTS { (u)-[:RATED]->(rec) }
  WITH rec, seed_movies,
       sum(top_a) AS score,
       collect(DISTINCT top_g.name) AS matched_genres
  RETURN rec.movieId AS movieId, rec.title AS title, rec.year AS year,
         score, matched_genres, seed_movies
  ORDER BY score DESC, title ASC
  LIMIT $inner_k
}
WITH collect({
  movieId: movieId, title: title, year: year,
  chain: head(matched_genres) + ' → ' + head(seed_movies) + ' → ' + title
}) AS genre_list
UNWIND range(0, size(genre_list) - 1) AS i
WITH genre_list, i
WITH collect({
  movieId: genre_list[i].movieId, title: genre_list[i].title, year: genre_list[i].year,
  strategy: 'genre', rank: i + 1, chain: genre_list[i].chain
}) AS genre_ranked

// Subquery 2 — Strategy 2 (director/actor)
CALL {
  MATCH (u:User {userId: $userId})-[r1:RATED]->(seed_d:Movie)-[:DIRECTED_BY]->(d:Director)
  WHERE r1.rating >= $threshold
  WITH u, d, count(seed_d) AS affinity, collect(DISTINCT seed_d.title) AS seed_titles
  ORDER BY affinity DESC
  LIMIT $top_n_directors
  WITH u, collect({
    person_id: elementId(d), name: d.name, kind: 'director',
    affinity: affinity, seeds: seed_titles
  }) AS top_dirs
  MATCH (u)-[r2:RATED]->(seed_a:Movie)-[fa:FEATURES_ACTOR]->(a:Actor)
  WHERE r2.rating >= $threshold AND fa.billing_order <= $max_billing
  WITH u, top_dirs, a, count(seed_a) AS affinity, collect(DISTINCT seed_a.title) AS seed_titles
  ORDER BY affinity DESC
  LIMIT $top_n_actors
  WITH u, top_dirs, collect({
    person_id: elementId(a), name: a.name, kind: 'actor',
    affinity: affinity, seeds: seed_titles
  }) AS top_actors
  WITH u, top_dirs + top_actors AS top_people
  UNWIND top_people AS tp
  MATCH (rec:Movie)
  WHERE NOT EXISTS { (u)-[:RATED]->(rec) }
    AND (
      (tp.kind = 'director' AND EXISTS {
        MATCH (rec)-[:DIRECTED_BY]->(d2:Director)
        WHERE elementId(d2) = tp.person_id
      })
      OR
      (tp.kind = 'actor' AND EXISTS {
        MATCH (rec)-[fa2:FEATURES_ACTOR]->(a2:Actor)
        WHERE elementId(a2) = tp.person_id AND fa2.billing_order <= $max_billing
      })
    )
  WITH rec, collect(tp) AS connecting_people
  WITH rec,
       [p IN connecting_people | p.name + ' (' + p.kind + ')'] AS via_people,
       reduce(s = 0, p IN connecting_people | s + p.affinity) AS score,
       [p IN connecting_people | head(p.seeds) + ' → ' + p.name + ' (' + p.kind + ') → ' + rec.title][..3] AS seed_chains
  RETURN rec.movieId AS movieId, rec.title AS title, rec.year AS year,
         score, via_people, seed_chains
  ORDER BY score DESC, title ASC
  LIMIT $inner_k
}
WITH genre_ranked, collect({
  movieId: movieId, title: title, year: year,
  chain: head(seed_chains)
}) AS da_list
UNWIND range(0, size(da_list) - 1) AS j
WITH genre_ranked, da_list, j
WITH genre_ranked, collect({
  movieId: da_list[j].movieId, title: da_list[j].title, year: da_list[j].year,
  strategy: 'director_actor', rank: j + 1, chain: da_list[j].chain
}) AS da_ranked

// Subquery 3 — Strategy 3 (collaborative)
CALL {
  MATCH (u:User {userId: $userId})-[r1:RATED]->(seed:Movie)<-[r2:RATED]-(peer:User)
  WHERE r1.rating >= $threshold
    AND r2.rating >= $threshold
    AND peer <> u
  WITH u, peer, count(DISTINCT seed) AS overlap, collect(DISTINCT seed.title) AS shared_titles
  WHERE overlap >= $min_overlap
  WITH u, peer, overlap, shared_titles
  ORDER BY overlap DESC
  LIMIT $top_n_peers
  WITH u, collect({
    peer_id: peer.userId, overlap: overlap, shared: shared_titles
  }) AS top_peers
  UNWIND top_peers AS tp
  MATCH (peer:User {userId: tp.peer_id})-[r3:RATED]->(rec:Movie)
  WHERE r3.rating >= $threshold
    AND NOT EXISTS { (u)-[:RATED]->(rec) }
  WITH rec, tp, r3.rating AS peer_rating
  WITH rec, collect({
    peer_id: tp.peer_id, overlap: tp.overlap,
    peer_rating: peer_rating, shared: tp.shared
  }) AS contributing
  WITH rec,
       reduce(s = 0.0, c IN contributing | s + (c.overlap * c.peer_rating)) AS score,
       [c IN contributing | {peer_id: c.peer_id, overlap: c.overlap}] AS via_peers,
       [c IN contributing | head(c.shared) + ' → peer ' + toString(c.peer_id) + ' → ' + rec.title][..3] AS peer_chains
  RETURN rec.movieId AS movieId, rec.title AS title, rec.year AS year,
         score, via_peers, peer_chains
  ORDER BY score DESC, title ASC
  LIMIT $inner_k
}
WITH genre_ranked, da_ranked, collect({
  movieId: movieId, title: title, year: year,
  chain: head(peer_chains)
}) AS cf_list
UNWIND range(0, size(cf_list) - 1) AS m
WITH genre_ranked, da_ranked, cf_list, m
WITH genre_ranked, da_ranked, collect({
  movieId: cf_list[m].movieId, title: cf_list[m].title, year: cf_list[m].year,
  strategy: 'collaborative', rank: m + 1, chain: cf_list[m].chain
}) AS cf_ranked

// Merge — union three rank-lists, group by movieId, weighted RRF score
WITH genre_ranked + da_ranked + cf_ranked AS all_results
UNWIND all_results AS r
WITH r.movieId AS movieId, r.title AS title, r.year AS year,
     collect({strategy: r.strategy, rank: r.rank, chain: r.chain}) AS contributions
WITH movieId, title, year, contributions,
     reduce(s = 0.0, c IN contributions |
       s + CASE c.strategy
             WHEN 'genre'          THEN $w_genre          / toFloat(c.rank)
             WHEN 'director_actor' THEN $w_director_actor / toFloat(c.rank)
             WHEN 'collaborative'  THEN $w_collab         / toFloat(c.rank)
           END
     ) AS score,
     [c IN contributions | c.strategy] AS via_strategies,
     [c IN contributions | c.chain]    AS chains
RETURN movieId, title, year, score, via_strategies, chains
ORDER BY score DESC, title ASC
LIMIT $k
"""

phase_4_rows = run_query(
    PHASE_4,
    {
        "userId": ANCHOR_USER, "threshold": 4.0,
        "top_n_genres": 3, "top_n_directors": 3, "top_n_actors": 5, "max_billing": 5,
        "min_overlap": 3, "top_n_peers": 25,
        "inner_k": INNER_K, "k": 10,
        "w_genre": 1.0, "w_director_actor": 1.0, "w_collab": 1.0,
    },
)

def render(rows):
    if not rows:
        return pd.DataFrame(columns=["title", "year", "score", "via_strategies", "chains"])
    df = pd.DataFrame(rows)
    df["score"] = df["score"].round(3)
    df["via_strategies"] = df["via_strategies"].apply(lambda xs: ", ".join(xs))
    df["chains"] = df["chains"].apply(lambda xs: "\n".join(xs))
    return df[["title", "year", "score", "via_strategies", "chains"]]

print(f"Phase 4 — {len(phase_4_rows)} fused recs (default weights 1.0/1.0/1.0):")
render(phase_4_rows)

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=3, column=1, offset=45>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 45, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n// Subquery 1 — Strategy 1 (genre affinity)\nCALL {\n  MATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)\n  WHERE r.rating >= $threshold\n  WITH u, g, count(*) AS affinity\n  ORDER BY affinity DESC\n  LIMIT $top_n_genres\n  WITH u, collect({g: g, a: affinity}) AS scored_genres\n  UNWIND scored_genres AS sg\n  WITH u

Phase 4 — 10 fused recs (default weights 1.0/1.0/1.0):


,title,year,score,via_strategies,chains
0,Aladdin,1992.0,1.000,director_actor,"Birdcage, The → Robin Williams (actor) → Aladdin"
1,Brassed Off,1996.0,1.000,genre,Comedy → Babe → Brassed Off
2,Star Wars,1977.0,1.000,collaborative,Babe → peer 303 → Star Wars
3,Aladdin and the King of Thieves,1996.0,0.500,director_actor,"Birdcage, The → Robin Williams (actor) → Aladd..."
4,Back to the Future,1985.0,0.500,collaborative,Babe → peer 303 → Back to the Future
5,Cinema Paradiso,1988.0,0.500,genre,Comedy → Babe → Cinema Paradiso
6,Being Human,1993.0,0.333,director_actor,"Birdcage, The → Robin Williams (actor) → Being..."
7,"Corrina, Corrina",1994.0,0.333,genre,"Comedy → Babe → Corrina, Corrina"
8,Dead Poets Society,1989.0,0.333,"director_actor, collaborative","Birdcage, The → Robin Williams (actor) → Dead ..."
9,Raiders of the Lost Ark,1981.0,0.333,collaborative,Babe → peer 303 → Raiders of the Lost Ark


## Now bring it all together

Cell 7's Cypher is exactly what [`src/recommender.py`](../src/recommender.py) assembles into `COMPOSITE_CYPHER` at module-load time — except the three subquery bodies aren't hand-pasted, they're built by string-substituting `$k` → `$inner_k` in the existing constants `GENRE_AFFINITY_CYPHER`, `DIRECTOR_ACTOR_CYPHER`, `COLLABORATIVE_CYPHER` and wrapping each in `CALL { ... }`. That's the trick that keeps the composite DRY: a fix to Strategy 1's body propagates to Strategy 4 for free.

Cell 9 prints the assembled constant. Cell 10 calls the wrapper function. The output should match Phase 4 above row-for-row — same `movieId`s, same scores, same order.

In [7]:
# 9. The unified constant — assembled at module-load from the three sub-strategy constants
from recommender import COMPOSITE_CYPHER
print(COMPOSITE_CYPHER)


// Subquery 1 — Strategy 1: genre affinity, top $inner_k
CALL {
// Phase 1 — user's top genres by count of >= threshold ratings
MATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)
WHERE r.rating >= $threshold
WITH u, g, count(*) AS affinity
ORDER BY affinity DESC
LIMIT $top_n_genres
WITH u, collect({g: g, a: affinity}) AS scored_genres

// Phase 2 — global seeds: user's >= threshold titles in those top genres (up to 5)
UNWIND scored_genres AS sg
WITH u, scored_genres, sg.g AS top_g
MATCH (u)-[r2:RATED]->(seed:Movie)-[:IN_GENRE]->(top_g)
WHERE r2.rating >= $threshold
WITH u, scored_genres, collect(DISTINCT seed.title)[..5] AS seed_movies

// Phase 3 — unseen recs in top genres, scored + ranked
UNWIND scored_genres AS sg
WITH u, seed_movies, sg.g AS top_g, sg.a AS top_a
MATCH (top_g)<-[:IN_GENRE]-(rec:Movie)
WHERE NOT EXISTS { (u)-[:RATED]->(rec) }
WITH rec, seed_movies,
     sum(top_a) AS score,
     collect(DISTINCT top_g.name) AS matched_genres
RETURN rec.movi

In [8]:
# 10. Call the wrapper — should match Phase 4 cell 7 row-for-row
from recommender import composite_multi_signal
recs = composite_multi_signal(ANCHOR_USER)
render(recs)

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=3, column=1, offset=58>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 58, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n// Subquery 1 — Strategy 1: genre affinity, top $inner_k\nCALL {\n// Phase 1 — user's top genres by count of >= threshold ratings\nMATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)\nWHERE r.rating >= $threshold\nWITH u, g, count(*) AS affinity\nORDER BY affinity DESC\nLIMIT $top_n_genres\nWITH u, collect({g: g, a: a

,title,year,score,via_strategies,chains
0,Aladdin,1992.0,1.000,director_actor,"Birdcage, The → Robin Williams (actor) → Aladdin"
1,Brassed Off,1996.0,1.000,genre,Comedy → Babe → Brassed Off
2,Star Wars,1977.0,1.000,collaborative,Babe → peer 303 → Star Wars
3,Aladdin and the King of Thieves,1996.0,0.500,director_actor,"Birdcage, The → Robin Williams (actor) → Aladd..."
4,Back to the Future,1985.0,0.500,collaborative,Babe → peer 303 → Back to the Future
5,Cinema Paradiso,1988.0,0.500,genre,Comedy → Babe → Cinema Paradiso
6,Being Human,1993.0,0.333,director_actor,"Birdcage, The → Robin Williams (actor) → Being..."
7,"Corrina, Corrina",1994.0,0.333,genre,"Comedy → Babe → Corrina, Corrina"
8,Dead Poets Society,1989.0,0.333,"director_actor, collaborative","Birdcage, The → Robin Williams (actor) → Dead ..."
9,Raiders of the Lost Ark,1981.0,0.333,collaborative,Babe → peer 303 → Raiders of the Lost Ark


## Reading the explainability chain

Pick the top rec from cell 10. Read its `via_strategies` first — a list telling you *which sub-strategies surfaced this rec*. A rec with `['genre', 'director_actor', 'collaborative']` is a three-way endorsement; a rec with just `['collaborative']` is a peer-only signal. The `score` (`sum(w_s / rank_s)`) reflects both the breadth (how many strategies picked it) and the depth (where in each ranked list it landed).

Then narrate one rec end-to-end. Suppose `via_strategies = ['genre', 'collaborative']` and `chains = ['Comedy → Some Movie → This Rec', 'Some Other Movie → peer 305 → This Rec']`:

> *"This rec hit Strategy 1's list because user 196 has rated Comedy films 4★+ — including **Some Movie** — and this rec is also Comedy. It also hit Strategy 3's list because **peer 305** (who overlaps with user 196 on many 4★+ titles, including **Some Other Movie**) rated this rec 4★+. The composite score reflects both signals: it scored `w_genre / rank_genre + w_collab / rank_collab` — for example `1.0/2 + 1.0/4 = 0.75` if it hit #2 in genre and #4 in collaborative."*

The contrast with Strategies 1 / 2 / 3 is the point of this notebook. Each of them carries **one** chain — genre-bridged, person-bridged, or peer-bridged. Strategy 4 carries **N chains**, one per contributing strategy, paired with `via_strategies` in the same order so you can `zip()` them safely. Honest per-strategy attribution without inventing a unified chain format.

Two more things to internalize:

1. **`score` is not commensurable across users.** A user with strong signal in all three strategies might see top scores around 3.0 (if a single rec hits #1 in every strategy). A thinner user might see top scores around 0.5. Same shape as 1 / 2 / 3 — Streamlit will need to either normalize per-user or display scores as "internal" if it surfaces them.
2. **`chains` order matches `via_strategies` order.** Both are built from the same `contributions` list in Cypher, in the same iteration order. Index 0 of `chains` is the chain for `via_strategies[0]`.

In [9]:
# 12. Knob — weights: skew the fusion toward one signal
genre_heavy = composite_multi_signal(ANCHOR_USER, w_genre=3.0, w_director_actor=1.0, w_collab=1.0)
collab_heavy = composite_multi_signal(ANCHOR_USER, w_genre=1.0, w_director_actor=1.0, w_collab=3.0)

print("w_genre=3.0 — top recs shift toward Strategy 1's high-rank entries:")
print(render(genre_heavy).head(5))
print("\nw_collab=3.0 — top recs shift toward Strategy 3's high-rank entries:")
print(render(collab_heavy).head(5))
print("\nThe weights are the 'I trust signal X more than signal Y' knob. Default (1.0, 1.0, 1.0)")
print("is neutral; reweighting is a deliberate choice that should be backed by an evaluation signal.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=3, column=1, offset=58>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 58, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n// Subquery 1 — Strategy 1: genre affinity, top $inner_k\nCALL {\n// Phase 1 — user's top genres by count of >= threshold ratings\nMATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)\nWHERE r.rating >= $threshold\nWITH u, g, count(*) AS affinity\nORDER BY affinity DESC\nLIMIT $top_n_genres\nWITH u, collect({g: g, a: a

w_genre=3.0 — top recs shift toward Strategy 1's high-rank entries:
              title    year  score  via_strategies  \
0       Brassed Off  1996.0    3.0           genre   
1   Cinema Paradiso  1988.0    1.5           genre   
2           Aladdin  1992.0    1.0  director_actor   
3  Corrina, Corrina  1994.0    1.0           genre   
4         Star Wars  1977.0    1.0   collaborative   

                                             chains  
0                       Comedy → Babe → Brassed Off  
1                   Comedy → Babe → Cinema Paradiso  
2  Birdcage, The → Robin Williams (actor) → Aladdin  
3                  Comedy → Babe → Corrina, Corrina  
4                       Babe → peer 303 → Star Wars  

w_collab=3.0 — top recs shift toward Strategy 3's high-rank entries:
                     title    year  score  via_strategies  \
0                Star Wars  1977.0    3.0   collaborative   
1       Back to the Future  1985.0    1.5   collaborative   
2                  Aladdin  19

In [10]:
# 13. Knob — inner_k: how deep into each sub-strategy do we look?
tight = composite_multi_signal(ANCHOR_USER, inner_k=5)
wide = composite_multi_signal(ANCHOR_USER, inner_k=50)

print("inner_k=5 — each sub-strategy contributes only its top 5; few recs survive to the merge:")
print(render(tight).head(5))
print(f"\n(total fused recs at inner_k=5: {len(tight)})")
print("\ninner_k=50 — tail positions (rank 20, 30, 40...) get into the merge but contribute tiny 1/rank values:")
print(render(wide).head(5))
print(f"\n(total fused recs at inner_k=50: {len(wide)})")
print("\nDefault inner_k=15 trades coverage against signal-to-noise. A larger inner_k surfaces more recs")
print("that hit the long tail of one strategy but the head of another — at the cost of letting weak")
print("rank-30+ entries dilute the score.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=3, column=1, offset=58>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 58, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n// Subquery 1 — Strategy 1: genre affinity, top $inner_k\nCALL {\n// Phase 1 — user's top genres by count of >= threshold ratings\nMATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)\nWHERE r.rating >= $threshold\nWITH u, g, count(*) AS affinity\nORDER BY affinity DESC\nLIMIT $top_n_genres\nWITH u, collect({g: g, a: a

inner_k=5 — each sub-strategy contributes only its top 5; few recs survive to the merge:
                             title    year  score  via_strategies  \
0                          Aladdin  1992.0    1.0  director_actor   
1                      Brassed Off  1996.0    1.0           genre   
2                        Star Wars  1977.0    1.0   collaborative   
3  Aladdin and the King of Thieves  1996.0    0.5  director_actor   
4               Back to the Future  1985.0    0.5   collaborative   

                                              chains  
0   Birdcage, The → Robin Williams (actor) → Aladdin  
1                        Comedy → Babe → Brassed Off  
2                        Babe → peer 303 → Star Wars  
3  Birdcage, The → Robin Williams (actor) → Aladd...  
4               Babe → peer 303 → Back to the Future  

(total fused recs at inner_k=5: 10)

inner_k=50 — tail positions (rank 20, 30, 40...) get into the merge but contribute tiny 1/rank values:
                         

In [11]:
# 14. Edge case — cold start + anchor sanity
# 14a. Non-existent user
n_ghost = len(composite_multi_signal(99999))
print(f"14a. User 99999 doesn't exist -> composite_multi_signal returned {n_ghost} recs.")

# 14b. A user with zero >=4* ratings (no signal in any sub-strategy), if one exists
no_signal_user = run_query(
    """
    MATCH (u:User)
    WHERE NOT EXISTS { MATCH (u)-[r:RATED]->() WHERE r.rating >= 4 }
    RETURN u.userId AS userId LIMIT 1
    """,
)
if no_signal_user:
    cold_id = no_signal_user[0]["userId"]
    n = len(composite_multi_signal(cold_id))
    print(f"14b. User {cold_id} has zero >=4* ratings -> composite_multi_signal returned {n} recs.")
else:
    print("14b. (No >=4*-free user in MovieLens 100K — skipping.)")

# 14c. Anchor sanity — none of the recs from cell 10 should already be rated by user 196
rec_ids = [r["movieId"] for r in recs]
overlap = run_query(
    """
    MATCH (u:User {userId: $userId})-[:RATED]->(m:Movie)
    WHERE m.movieId IN $ids
    RETURN count(m) AS overlap
    """,
    {"userId": ANCHOR_USER, "ids": rec_ids},
)
n = overlap[0]["overlap"] if overlap else -1
print(f"14c. {'OK — 0 overlap' if n == 0 else f'MISMATCH — user 196 has already rated {n} of these recs'}")

print("\nContract: empty list = cold start (all three sub-strategies returned []).")
print("Demographic-pivot fallback (the 'user with 1 rating still gets recs' teaching moment)")
print("is filed as T-22b, not delivered by T-22.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=3, column=1, offset=58>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 58, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n// Subquery 1 — Strategy 1: genre affinity, top $inner_k\nCALL {\n// Phase 1 — user's top genres by count of >= threshold ratings\nMATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)\nWHERE r.rating >= $threshold\nWITH u, g, count(*) AS affinity\nORDER BY affinity DESC\nLIMIT $top_n_genres\nWITH u, collect({g: g, a: a

14a. User 99999 doesn't exist -> composite_multi_signal returned 0 recs.


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=3, column=1, offset=58>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 58, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n// Subquery 1 — Strategy 1: genre affinity, top $inner_k\nCALL {\n// Phase 1 — user's top genres by count of >= threshold ratings\nMATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)\nWHERE r.rating >= $threshold\nWITH u, g, count(*) AS affinity\nORDER BY affinity DESC\nLIMIT $top_n_genres\nWITH u, collect({g: g, a: a

14b. User 685 has zero >=4* ratings -> composite_multi_signal returned 0 recs.
14c. OK — 0 overlap

Contract: empty list = cold start (all three sub-strategies returned []).
Demographic-pivot fallback (the 'user with 1 rating still gets recs' teaching moment)
is filed as T-22b, not delivered by T-22.


In [12]:
# 15. Cleanup
close_driver()
print("Driver closed.")

Driver closed.
